In [1]:
import numpy as np
import scipy.stats as stats

def compute_ci(data, alpha = 0.05):
    n = len(data)
    mean = np.mean(data)
    std_err = np.std(data, ddof = 1) / np.sqrt(n)
    t_crit = stats.t.ppf(1 - alpha/2, df = n-1)
    lower = mean - t_crit * std_err
    upper = mean + t_crit *std_err
    return mean, lower, upper


# Ex1

In [2]:
# Monte carlo
np.random.seed(42)
n = 100
U = np.random.uniform(0,1,n)

Xi = np.exp(U)

X = np.sum(Xi)/n
mean_monte, lower_monte, upper_monte = compute_ci(Xi)

X, lower_monte, upper_monte

(np.float64(1.6720612537554935),
 np.float64(1.5728239057161093),
 np.float64(1.7712986017948777))

# Ex 2

In [3]:
# Antithetic variables

eU = np.exp(U)

Yi = (eU+np.exp(1)/eU)/2

Y = np.sum(Yi)/n

mean_A, lower_A, upper_A = compute_ci(Yi)

Y, lower_A, upper_A

(np.float64(1.7226021253193073),
 np.float64(1.7101879016931159),
 np.float64(1.7350163489454988))

# Ex 3

In [4]:
# Control variates

c = -np.cov(Xi, U)[0,1]/np.var(U)
mu = 0.5
Yi = Xi + c*(U - mu)

Y = np.sum(Yi)/n


mean_C, lower_C, upper_C = compute_ci(Yi)

Y, lower_C, upper_C

(np.float64(1.7223070150700954),
 np.float64(1.7099399890438187),
 np.float64(1.7346740410963721))

# Ex 4

In [5]:
k = 10
n_4 = int(n/k)
Yi = []

for i in range(1,k+1):
    Uij = np.random.uniform((i-1)/10,i/10,n_4)
    eUij = np.exp(Uij)
    Yi.append(np.sum(eUij)/k)

Yi = np.array(Yi)

Y = np.sum(Yi)/n_4


mean_S, lower_S, upper_S = compute_ci(Yi)

Y, lower_S, upper_S

(np.float64(1.7192217442637012),
 np.float64(1.3450686903225462),
 np.float64(2.0933747982048563))

# Ex 5

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import math

m = 10
mean_time = 1
mean_service = 8
n_reps = 10
n_customers = 10000
np.random.seed(42)


def simulate_loss_system(generate_arrivals, generate_services):
    blocking_fraction = []
    avg_inter_arrivals = []

    for _  in range(n_reps):
        inter_arrivals = generate_arrivals(n_customers)
        service_times = generate_services(n_customers)
        arrival_times = np.cumsum(inter_arrivals)

        servers_free_time = np.zeros(m)
        blocked = 0

        for i in range(n_customers):
            arr = arrival_times[i]

            free_servers = servers_free_time <= arr

            if np.any(free_servers):
                idx = np.argmax(free_servers)
                servers_free_time[idx] = arr + service_times[i]
            else:
                blocked += 1

        blocking_fraction.append(blocked / n_customers)
        avg_inter_arrivals.append(np.mean(inter_arrivals))
    
    return np.array(blocking_fraction), np.array(avg_inter_arrivals)

mu_z = 1

def arr_poisson(n):
    return np.random.exponential(scale=mean_time, size = n)

def srv_exp(n):
    return np.random.exponential(scale = mean_service, size = n)


Y_block, Z_arr = simulate_loss_system(arr_poisson, srv_exp)

cov_mat = np.cov(Y_block,Z_arr)
c_star_queue = -cov_mat[0,1] / cov_mat[1,1]
y_cv_queue = Y_block + c_star_queue * (Z_arr - mu_z)

mean_mm, low_mm, up_mm = compute_ci(y_cv_queue)

print(f"Simulated Blocking: {mean_mm:.4f}  (95% CI: [{low_mm:.4f}, {up_mm:.4f}])")


Simulated Blocking: 0.1230  (95% CI: [0.1191, 0.1270])


# Part 6

In [41]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import math

m = 10
mean_time = 1
mean_service = 8
n_reps = 10
n_customers = 10000
np.random.seed(42)

p1 = 0.8
lambda1 = 0.8333
p2 = 0.2
lambda2 = 5


def simulate_loss_system(inter_arrivals, service_times):
    blocking_fraction = []
    avg_inter_arrivals = []

    for _  in range(n_reps):

        arrival_times = np.cumsum(inter_arrivals)

        servers_free_time = np.zeros(m)
        blocked = 0

        for i in range(n_customers):
            arr = arrival_times[i]

            free_servers = servers_free_time <= arr

            if np.any(free_servers):
                idx = np.argmax(free_servers)
                servers_free_time[idx] = arr + service_times[i]
            else:
                blocked += 1

        blocking_fraction.append(blocked / n_customers)
        avg_inter_arrivals.append(np.mean(inter_arrivals))
    
    return np.array(blocking_fraction), np.array(avg_inter_arrivals)

Y_block_poisson = []
Y_block_hyper = []

for _ in range(n_reps):
    u_arr_crn = np.random.rand(n_customers)
    u_srv_crn = np.random.rand(n_customers)
    u_coin_crn = np.random.rand(n_customers)

    arrivals_poisson = -(1/1) * np.log(u_arr_crn) 

    arrivals_hyper = np.where(u_coin_crn <= p1, 
                              -(1 / lambda1) * np.log(u_arr_crn),
                              -(1 / lambda2) * np.log(u_arr_crn))
    

    service_exp_times = -mean_time * np.log(u_srv_crn)

    blocking_poisson = simulate_loss_system(arrivals_poisson, service_exp_times,)
    blocking_hyper = simulate_loss_system(arrivals_hyper, service_exp_times)

    Y_block_poisson.append(blocking_poisson)
    Y_block_hyper.append(blocking_hyper)


diff = np.array(Y_block_poisson) - np.array(Y_block_hyper)

diff.mean(), diff.var(ddof=1), compute_ci(Y_block_poisson), compute_ci(Y_block_hyper)

(np.float64(-0.001064954144779806),
 np.float64(1.4103519546067166e-05),
 (np.float64(0.49998710745811364),
  np.float64(0.14137097027865292),
  np.float64(0.8586032446375744)),
 (np.float64(0.5010520616028935),
  np.float64(0.14167682370356116),
  np.float64(0.8604272995022259)))

# Ex 7

In [44]:
# Monte carlo
np.random.seed(42)
n = 100000
a2 = 2
a4 = 4
sigma = 1

Z2 = np.random.normal(a2, sigma, size = n)
Z4 = np.random.normal(a4, sigma, size = n)


MC_est_4 = np.size(np.where(Z4 > a4))/n
MC_est_2 = np.size(np.where(Z2 > a2))/n
MC_est_2, MC_est_4


(0.50112, 0.50015)

In [50]:
# Monte carlo
np.random.seed(42)
n = 10000
a2 = 2
a4 = 4
sigma = 1

Z = np.random.normal(0, 1, size = n)

MC_est_2 = np.size(np.where(Z > a2))/n
MC_est_4 = np.size(np.where(Z > a4))/n


MC_est_2, MC_est_4


(0.0237, 0.0)

In [53]:
def importance_samp(a, s, n):
    sig = np.sqrt(s)

    x = np.random.normal(a, sig, size = n)

    f = stats.norm.pdf(x, loc=0, scale=1)
    g = stats.norm.pdf(x, loc=a, scale=sig)

    w = f/g

    values = w*(x > a)

    theta_IS = np.mean(values)

    return theta_IS


importance_samp(a2, sigma, n), importance_samp(a4, sigma, n)

(np.float64(0.022708316182480523), np.float64(3.273009339580869e-05))

# Ex 8

In [67]:
n = 100000
np.random.seed(42)

def importance_samp_e(n):
    lambda_ = np.linspace(0.1, 10, 1000)

    var = []

    for i in range(len(lambda_)):
        x = np.random.exponential(scale = 1/lambda_[i], size = n)

        f = (x >= 0) & (x <= 1)
        hf = np.exp(x) * f
        g = lambda_[i]*np.exp(-lambda_[i]*x)

        values = hf/g

        theta_IS = np.mean(values)
        var_IS = np.var(values, ddof = 1)

        var.append(var_IS)

    mini = np.argmin(var)

    mini_var = var[mini]
    mini_lambda_ = lambda_[mini]

    return mini_var, mini_lambda_

importance_samp_e(n)


(np.float64(3.099934345054741), np.float64(1.3783783783783785))